# 🚀 RAPID v0.2.0 — LCA Clustering (Local Development & Validation)

Versão adaptada de `RAPIDv0_2_0_LCA_Clustering_sampleSINAN.ipynb` para desenvolvimento local e validação da funcionalidade de **Latent Class Analysis (LCA)**.

Principais adaptações:
1. **Pacote local**: utiliza `isaric` instalado em modo editável (`pip install -e .`) a partir do repositório local (sem `pip install git+https://...` e sem dependência obrigatória do Google Colab).
2. **Diagnóstico automático do ambiente**: valida origem do pacote `isaric.__file__`, versão do Python, branch/commit Git e versões das dependências.
3. **Objetos de modelagem independentes**: separa `model_no_bootstrap` (sem bootstrap) e `model_bootstrap` (com bootstrap) para evitar sobrescrita de resultados.
4. **Isolamento de artefatos**: salva relatórios e modelos em `artifacts/lca_baseline/notebook_outputs/` (ignorado pelo Git), mantendo os diretórios de código limpos.

## 🔧 1. Diagnóstico do Ambiente e Importação Local

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys
import time

import numpy as np
import pandas as pd

# Localiza a raiz do repositório independentemente do diretório atual do kernel
def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "isaric").exists():
            return candidate
    raise RuntimeError("Não foi possível localizar a raiz do repositório RAPID.")

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import isaric as isa
from tests.lca.synthetic_data import LCA_SYMPTOM_VARS, generate_synthetic_lca_data

In [ ]:
# Validação automática de que isaric está sendo carregado do repositório local e da feature branch
expected_init = (REPO_ROOT / "src" / "isaric" / "__init__.py").resolve()
actual_init = Path(isa.__file__).resolve()
git_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO_ROOT, text=True).strip()
git_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True).strip()

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version.split()[0]}")
print(f"Git branch/commit : {git_branch} ({git_commit})")
print(f"RAPID version     : {isa.__version__}")
print(f"isaric.__file__   : {actual_init}")
print(f"Repo root         : {REPO_ROOT}")

assert actual_init == expected_init, (
    f"ERRO: isaric importado de {actual_init}, esperado {expected_init}. "
    f"Execute 'python -m pip install -e .' no ambiente local."
)
print("✅ Validação confirmada: isaric carregado diretamente do código local!")

for pkg in ["pandas", "numpy", "scipy", "scikit-learn", "stepmix", "matplotlib", "plotly", "reportlab"]:
    print(f"  - {pkg}: {importlib.metadata.version(pkg)}")

## 📥 2. Carregamento de Dados e Variáveis Clínicas

Procura o arquivo `DenguePos_child_Sample150.feather` em caminhos relativos ao repositório (`data/`, raiz) ou na pasta local `~/Downloads`. Caso não esteja disponível (ou se `RAPID_LCA_FORCE_SYNTHETIC=1`), utiliza o gerador determinístico de dados sintéticos binários (`seed=42`) com as mesmas 13 variáveis clínicas.

In [ ]:
MEASUREMENT_VARS = [
    "Fever",
    "Myalgia",
    "Headache",
    "Rash",
    "Vomiting",
    "Nausea",
    "Back pain",
    "Conjunctivitis",
    "Arthritis",
    "Arthralgia",
    "Petechiae",
    "Tourniquet test",
    "Retro-orbital pain",
]

# Caminho externo opcional configurável via variável de ambiente RAPID_LCA_DATA_PATH
env_data_path = os.environ.get("RAPID_LCA_DATA_PATH")
CANDIDATE_DATA_PATHS = []
if env_data_path:
    CANDIDATE_DATA_PATHS.append(Path(env_data_path).expanduser())
CANDIDATE_DATA_PATHS.extend([
    REPO_ROOT / "data" / "DenguePos_child_Sample150.feather",
    REPO_ROOT / "DenguePos_child_Sample150.feather",
    Path.home() / "Downloads" / "DenguePos_child_Sample150.feather",
])

force_synthetic = os.environ.get("RAPID_LCA_FORCE_SYNTHETIC", "0") == "1"
feather_path = None
if not force_synthetic:
    for candidate in CANDIDATE_DATA_PATHS:
        if candidate.exists():
            feather_path = candidate
            break

if feather_path is not None:
    df = pd.read_feather(feather_path)
    DATA_SOURCE = f"clinical_feather ({feather_path.name})"
    print(f"✅ Dataset clínico carregado: {feather_path.name} | shape={df.shape}")
else:
    df = generate_synthetic_lca_data(n_samples=150, seed=42, symptom_vars=MEASUREMENT_VARS)
    DATA_SOURCE = "synthetic_generator (n=150, seed=42)"
    print(f"ℹ️ Usando dataset sintético para validação técnica: shape={df.shape}")

# Verificação automática das variáveis binárias de sintomas (sem exibir registros individuais sensíveis)
missing_vars = [v for v in MEASUREMENT_VARS if v not in df.columns]
assert not missing_vars, f"Variáveis ausentes no DataFrame: {missing_vars}"
print(f"✅ Todas as {len(MEASUREMENT_VARS)} variáveis clínicas de sintomas estão presentes e validadas.")

## 📈 3. Cenário A — LCA Sem Bootstrap (`model_no_bootstrap`)

Treinamento com enumeração de classes sem bootstrap (`bootstrap=False`).
Utiliza objeto independente `model_no_bootstrap`.

In [ ]:
FAST_MODE = os.environ.get("RAPID_LCA_FAST_TEST", "0") == "1"
K_RANGE_NO_BOOT = [1, 2, 3, 4] if FAST_MODE else list(range(1, 16))
REF_K_NO_BOOT = max(K_RANGE_NO_BOOT)

t0 = time.perf_counter()
model_no_bootstrap = isa.RAPID.create(
    data=df,
    model="lca",
    n_clusters=REF_K_NO_BOOT,
    measurement_vars=MEASUREMENT_VARS,
    random_seed=42,
)

model_no_bootstrap.fit(
    bootstrap=False,
    param_grid={"n_clusters": K_RANGE_NO_BOOT},
)
elapsed_no_boot = time.perf_counter() - t0

print(f"✅ model_no_bootstrap treinado em {elapsed_no_boot:.2f}s (estado: {model_no_bootstrap._state})")
print(f"   AIC: {model_no_bootstrap.metrics.get('aic', float('nan')):.3f}")
print(f"   BIC: {model_no_bootstrap.metrics.get('bic', float('nan')):.3f}")
print(f"   Entropy: {model_no_bootstrap.metrics.get('entropy', float('nan')):.3f}")

# Verificações automáticas do Cenário A
assert model_no_bootstrap._state == "fitted"
assert model_no_bootstrap.grid_results is not None
assert list(model_no_bootstrap.grid_results["n_clusters"]) == K_RANGE_NO_BOOT
assert sorted(model_no_bootstrap.model_objects.keys()) == K_RANGE_NO_BOOT
grid_snapshot_no_boot = model_no_bootstrap.grid_results.copy()

In [ ]:
# 1. Summary padrão sem bootstrap (LCA-03 + GEN-03: exibe tabelas essenciais e gráficos padrão incluindo radar_plot)
model_no_bootstrap.summary()
assert "radar_plot" in model_no_bootstrap._displayed_plots

# 2. Exploração e comparação de soluções com diferentes k sem retreinar (LCA-06 + LCA-07)
model_no_bootstrap.summary(k=2)
model_no_bootstrap.summary(k=3)
model_no_bootstrap.summary(k=3, plots=["radar_plot"])
model_no_bootstrap.summary(k=2, plots=["profile_heatmap"])
model_no_bootstrap.summary(k=3, table_format="short", plots=[])

# 3. Retorno ao summary padrão da solução de referência e verificação da tabela completa de enumeração
model_no_bootstrap.summary()
assert model_no_bootstrap.fitted_model.n_components == REF_K_NO_BOOT
assert list(model_no_bootstrap.grid_results["n_clusters"]) == K_RANGE_NO_BOOT
assert len(model_no_bootstrap._displayed_plots) == 5

## 🔄 4. Cenário B — LCA Com Bootstrap (`model_bootstrap`)

Treinamento com bootstrap (`bootstrap=True`) utilizando um objeto separado `model_bootstrap`, garantindo que `model_no_bootstrap` não seja sobrescrito.

In [ ]:
K_RANGE_BOOT = [1, 2, 3, 4] if FAST_MODE else list(range(1, 11))
REF_K_BOOT = max(K_RANGE_BOOT)
N_BOOT_ITER = 5 if FAST_MODE else 20

t0 = time.perf_counter()
model_bootstrap = isa.RAPID.create(
    data=df,
    model="lca",
    n_clusters=REF_K_BOOT,
    measurement_vars=MEASUREMENT_VARS,
    random_seed=42,
)

model_bootstrap.fit(
    bootstrap=True,
    n_iterations=N_BOOT_ITER,
    param_grid={"n_clusters": K_RANGE_BOOT},
)
elapsed_boot = time.perf_counter() - t0

print(f"🎉 model_bootstrap treinado em {elapsed_boot:.2f}s (n_iterations={N_BOOT_ITER})")

# Verificação automática de independência entre model_no_bootstrap e model_bootstrap
assert model_no_bootstrap is not model_bootstrap
pd.testing.assert_frame_equal(model_no_bootstrap.grid_results, grid_snapshot_no_boot)
assert model_no_bootstrap.lrt_results["blrt"] is None
assert model_bootstrap.lrt_results["blrt"] is not None
print("✅ Independência confirmada: model_no_bootstrap permaneceu intacto após treinar model_bootstrap!")

In [ ]:
# Summary padrão com bootstrap (exibe automaticamente os gráficos padrão + bootstrap_metrics_ic, bootstrap_metrics_entropy e selection_rate)
model_bootstrap.summary()
assert "radar_plot" in model_bootstrap._displayed_plots
assert "bootstrap_metrics_ic" in model_bootstrap._displayed_plots
assert "bootstrap_metrics_entropy" in model_bootstrap._displayed_plots

## 💾 5. Persistência e Relatório em Diretório Isolado (`artifacts/lca_baseline/notebook_outputs`)

Para evitar gerar arquivos `.rapid`, `.csv`, `.png` ou `.pdf` dentro dos diretórios de código-fonte, a exportação é direcionada para `artifacts/lca_baseline/notebook_outputs/`.

In [ ]:
OUTPUT_DIR = REPO_ROOT / "artifacts" / "lca_baseline" / "notebook_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prev_cwd = Path.cwd()
try:
    os.chdir(OUTPUT_DIR)
    model_bootstrap.decide(
        selected=True,
        k=3,
        reason="Baseline local development test for k=3 decision recording.",
    )
    model_bootstrap.report(format=["csv", "png", "pdf"])
finally:
    os.chdir(prev_cwd)

print(f"✅ Relatórios gerados em: {OUTPUT_DIR.relative_to(REPO_ROOT)}")